In [ ]:
import io
import gc
import pymupdf as fitz
import torch
import base64
import transformers
from PIL import Image
import pypdfium2 as pdfium
from dotenv import load_dotenv

from huggingface_hub.utils import enable_progress_bars
from colpali_engine.models import ColQwen2, ColQwen2Processor

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

In [ ]:
load_dotenv()

In [ ]:
pdf_path = r"C:\Users\UserAdmin\Documents\Multimodal-RAG\pdfs\government-personal-data-protection-policies-jul21.pdf"

In [ ]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()

In [ ]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [ ]:
def get_coarse_embedding(embeddings_tensor):
    pooled = embeddings_tensor.mean(dim=1) # mean pooling 
    normalised = pooled / pooled.norm(dim=-1, keepdim=True) # L2 normalization 
    return normalised.squeeze().to(torch.float32).cpu()

In [ ]:
def embed(pdf_path, model, processor):
    vector_index = []
    fitz_doc = fitz.open(pdf_path)
    for i in range(len(fitz_doc)):
        page = fitz_doc.load_page(i)
        pix = page.get_pixmap(dpi=150)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

        inputs = processor.process_images(images=[img])
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            embeddings = model(**inputs)
            embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True) #L2 normalization 
            coarse_vector = get_coarse_embedding(embeddings).flatten().tolist()
            embeddings = embeddings.squeeze(0).to(torch.float32).cpu()

        vector_index.append({
            "page_number": i+1,
            "coarse_vector": coarse_vector,
            "page_embeddings": embeddings
        })
    return vector_index

        

    
    

In [1]:
import torch
print("Is GPU available?:", torch.cuda.is_available())
print("Current Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


print(torch.__version__)      # If it ends in '+cpu', you have the wrong version.
print(torch.version.cuda)     # If this is None, CUDA support is not compiled in.


Is GPU available?: True
Current Device: AMD Radeon RX 7700 XT
2.9.1+rocm7.2.1
None


In [ ]:
vector_index = embed(pdf_path, model, processor)
print(vector_index)

In [ ]:
vector_index[0]['page_embeddings']